In [9]:
import json
import os
from pathlib import Path
import pandas as pd
import glob
from tqdm import tqdm
from PIL import Image
import numpy as np

### Frame select entries

In [ ]:
def extract_text_from_qa_frame_select(json_path):
    """
    Extract text content from message_list for all roles in the JSON data.
    
    Args:
        json_data: Dictionary containing the parsed JSON data
        
    Returns:
        List of dictionaries with role and text content
    """
    with open(json_path, 'r') as f:
        json_data = json.load(f)

    text_content = {"prompt": "", "subtask_target": "", "keyframe_1": ""}
    
    if 'message_list' in json_data:
        for message in json_data['message_list']:
            role = message.get('role')
            content_list = message.get('content')
            if role == 'system':
                text_content["prompt"] += content_list[0]['text'] + '\n\n'
            elif role == 'user':
                text_content["prompt"] += content_list[0]['text'] + '\n\n'
                if 'video' in content_list[1]:
                    text_content["keyframe_1"] = content_list[1]['video'][-1] # TODO: just picking one to test for now
            elif role == 'assistant':
                text_content["subtask_target"] = "\n\n<tool_call>{\"name\": \"select_frames\", \"arguments\": {\"target_frames\": [1]}}</tool_call>" # TODO: hardcoded because only providing one frame for now, always picking that frame as useful
                break
    
    return text_content

example_json_path = "/iris/u/jrpan/robo_memory/datasets/split_keyframes_subsample_5_retry_05_new_fs_prompt_new_golden_frames_image_aug/2025-07-03__Fri_Jul__4_00_25_17_2025_keyframes/qa_data/qa_frame_select_059.json"

extracted_messages = extract_text_from_qa_frame_select(example_json_path)
df = pd.DataFrame([extracted_messages])
df

,prompt,subtask_target,keyframe_1
0,You are a helpful assistant trying to instruct...,"\n\n<tool_call>{""name"": ""select_frames"", ""argu...",/iris/u/jrpan/robo_memory/datasets/split_keyfr...


### Primitive prediction entries

In [10]:
def extract_text_from_qa_primitive_prediction(json_path):
    with open(json_path, 'r') as f:
        json_data = json.load(f)

    text_content = {"prompt": ""}
    
    if 'message_list' in json_data:
        for message in json_data['message_list']:
            role = message.get('role')
            content_list = message.get('content')
            for content in content_list:
                if 'text' in content and (role == 'user' or role == 'system'):
                    text_content["prompt"] += content['text'] + '\n\n'
                elif 'video' in content and role == 'user':
                    video_frames = content['video']
                    # Take up to the last 4 frames
                    last_frames = video_frames[-4:]
                    for i, frame_path in enumerate(last_frames, 1):
                        text_content[f"recent_frame_{i}_path"] = frame_path
                elif 'image' in content and role == 'user':
                    keyframe_index = 1
                    while f"keyframe_{keyframe_index}_path" in text_content and keyframe_index <= 3:
                        keyframe_index += 1
                    # Take up to the first 3 keyframes
                    if keyframe_index <= 3:
                        text_content[f"keyframe_{keyframe_index}_path"] = content['image']
                elif 'text' in content and role == 'assistant':
                    text_content["subtask_target"] = content['text']
                    break
            
    
    return text_content

def load_images_from_paths(extracted_messages, image_keys):
    """
    Load images from file paths and convert them to numpy arrays.
    
    Args:
        extracted_messages: Dictionary containing image file paths
        image_keys: List of keys in extracted_messages that contain image file paths (format: keyframe_*_path, recent_frame_*_path)
        
    Returns:
        Dictionary with image arrays instead of file paths
    """
    for key in image_keys:
        if key in extracted_messages:
            image_path = extracted_messages[key]
            image = Image.open(image_path).convert('RGB')
            image_array = np.array(image)
            new_key = key.replace("_path", "")
            extracted_messages[new_key] = image_array
    
    return extracted_messages

In [ ]:
qa_data_dir = "/iris/u/jrpan/robo_memory/datasets/split_keyframes_subsample_5_retry_05_new_fs_prompt_new_golden_frames_image_aug/2025-07-03__Fri_Jul__4_00_25_17_2025_keyframes/qa_data"

qa_answer_files = glob.glob(os.path.join(qa_data_dir, "qa_answer_*.json"))
print(f"Found {len(qa_answer_files)} qa_answer_*.json files")

all_extracted_data = []
for json_file in tqdm(qa_answer_files):
    try:
        extracted_messages = extract_text_from_qa_primitive_prediction(json_file)
        # Pass all keys that match the keyframe_*_path and recent_frame_*_path pattern
        image_keys = [key for key in extracted_messages.keys() if key.endswith('_path') and (key.startswith('keyframe_') or key.startswith('recent_frame_'))]
        extracted_messages = load_images_from_paths(extracted_messages, image_keys)
        all_extracted_data.append(extracted_messages)
    except Exception as e:
        print(f"Error processing {os.path.basename(json_file)}: {e}")

df = pd.DataFrame(all_extracted_data)
print(f"Created DataFrame with {len(df)} rows")
df.head()

Found 655 qa_answer_*.json files


100%|██████████| 655/655 [00:08<00:00, 76.57it/s]


Created DataFrame with 655 rows


,prompt,recent_frame_1_path,recent_frame_2_path,recent_frame_3_path,recent_frame_4_path,keyframe_1_path,keyframe_2_path,keyframe_3_path,subtask_target,recent_frame_1,recent_frame_2,recent_frame_3,recent_frame_4,keyframe_1,keyframe_2,keyframe_3
0,You are a helpful assistant trying to instruct...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,pick up the fried chicken and put it in the wh...,"[[[70, 69, 67], [68, 69, 67], [68, 69, 67], [6...","[[[47, 47, 41], [45, 45, 41], [43, 41, 39], [4...","[[[42, 39, 35], [43, 39, 36], [44, 40, 39], [4...","[[[45, 43, 41], [45, 42, 41], [45, 42, 41], [4...","[[[38, 36, 34], [45, 42, 41], [48, 47, 44], [4...","[[[204, 196, 194], [205, 196, 198], [207, 200,...","[[[235, 229, 223], [255, 255, 255], [255, 249,..."
1,You are a helpful assistant trying to instruct...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,pick up the eraser and put it in the white bin,"[[[46, 46, 44], [38, 36, 35], [28, 28, 25], [2...","[[[38, 38, 39], [40, 40, 41], [40, 40, 41], [4...","[[[61, 61, 60], [61, 61, 59], [60, 60, 57], [6...","[[[78, 79, 73], [79, 79, 74], [79, 79, 75], [7...","[[[75, 74, 74], [67, 65, 65], [57, 55, 55], [5...","[[[195, 189, 186], [199, 192, 188], [205, 197,...","[[[191, 188, 181], [173, 165, 153], [126, 113,..."
2,You are a helpful assistant trying to instruct...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,pick up the red block and put it in the white bin,"[[[73, 66, 67], [73, 67, 66], [72, 66, 66], [7...","[[[19, 16, 14], [21, 18, 16], [19, 16, 14], [1...","[[[51, 45, 40], [50, 44, 40], [48, 43, 38], [4...","[[[79, 74, 70], [79, 74, 72], [78, 73, 70], [7...","[[[75, 74, 74], [67, 65, 65], [57, 55, 55], [5...","[[[195, 189, 186], [199, 192, 188], [205, 197,...","[[[191, 188, 181], [173, 165, 153], [126, 113,..."
3,You are a helpful assistant trying to instruct...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,NaN,NaN,NaN,look inside the white bin,"[[[54, 54, 53], [55, 53, 53], [55, 53, 53], [5...","[[[56, 56, 53], [57, 55, 54], [57, 57, 57], [5...","[[[47, 47, 47], [34, 34, 34], [23, 23, 22], [2...","[[[66, 64, 63], [67, 65, 64], [67, 65, 64], [6...",NaN,NaN,NaN
4,You are a helpful assistant trying to instruct...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,/iris/u/jrpan/robo_memory/datasets/split_keyfr...,NaN,NaN,look inside the black bin,"[[[32, 30, 29], [31, 30, 29], [29, 28, 27], [2...","[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ...","[[[43, 39, 38], [45, 41, 40], [46, 43, 41], [4...","[[[37, 35, 38], [38, 39, 41], [41, 42, 44], [4...","[[[41, 42, 44], [43, 44, 45], [36, 35, 37], [2...",NaN,NaN


In [11]:
df[['prompt', 'subtask_target', 'keyframe_1', 'keyframe_2', 'keyframe_3', 'recent_frame_1', 'recent_frame_2', 'recent_frame_3', 'recent_frame_4']].to_pickle("/iris/u/jrpan/openpi/subtask_prediction_df_tmp_longer.pkl")